In [3]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from PIL import Image
from io import BytesIO
from transformers import CLIPProcessor, CLIPModel
import tkinter as tk
from tkinter import ttk
from PIL import ImageTk, Image

def scrape_images_from_url_in_memory(url):
    """
    Scrapes all images from a given URL and processes them in memory without saving to disk.

    :param url: URL of the webpage to scrape images from.
    :return: List of tuples (Image object, URL).
    """
    try:
        response = requests.get(url)
        response.raise_for_status()

        soup = BeautifulSoup(response.text, 'html.parser')
        img_tags = soup.find_all('img')

        print(f"Found {len(img_tags)} images. Processing in memory...")

        images = []
        for img_tag in img_tags:
            img_url = img_tag.get('src')
            if not img_url:
                continue

            img_url = urljoin(url, img_url)

            if img_url.lower().endswith('.svg'):
                continue

            try:
                img_response = requests.get(img_url, stream=True)
                img_response.raise_for_status()
                img = Image.open(BytesIO(img_response.content))
                images.append((img, img_url))
            except Exception as e:
                print(f"Failed to process {img_url}: {e}")

        return images
    except requests.exceptions.RequestException as e:
        print(f"Failed to fetch URL: {e}")
        return []

def get_top_results(text, images, top_n=5):
    """
    Returns the top N images ranked by similarity to the input text.

    :param text: Input text for similarity ranking.
    :param images: List of tuples (Image object, URL).
    :param top_n: Number of top results to return.
    :return: List of tuples (Image object, URL, score).
    """
    model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
    processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

    pil_images = [img for img, _ in images]
    inputs = processor(text=[text], images=pil_images, return_tensors="pt", padding=True)

    outputs = model(**inputs)
    logits_per_image = outputs.logits_per_image
    probs = logits_per_image.softmax(dim=0).squeeze().tolist()

    if isinstance(probs, float):
        probs = [probs]

    sorted_indices = sorted(range(len(probs)), key=lambda i: probs[i], reverse=True)
    return [(images[idx][0], images[idx][1], probs[idx]) for idx in sorted_indices[:top_n]]

def create_gui():
    """Creates a GUI for image and video search."""
    def search_action():
        url = url_input.get()
        text = text_input.get("1.0", "end-1c")
        if url:
            scraped_images = scrape_images_from_url_in_memory(url)
            if scraped_images:
                results = get_top_results(text, scraped_images)
                display_results(results)

    def display_results(results):
        """Displays the top search results in the GUI."""
        for widget in results_frame.winfo_children():
            widget.destroy()

        cols = len(results)
        for col, (img, url, score) in enumerate(results):
            frame = tk.Frame(results_frame, padx=5, pady=5)
            frame.grid(row=0, column=col, sticky="n")

            img.thumbnail((150, 150))
            img_tk = ImageTk.PhotoImage(img)

            img_label = tk.Label(frame, image=img_tk)
            img_label.image = img_tk  # Keep a reference to avoid garbage collection
            img_label.pack()

            text_label = tk.Label(frame, text=f"Score: {score:.2f}\nURL: {url[:30]}...", font=("Helvetica", 10))
            text_label.pack()

    # Initialize GUI window
    root = tk.Tk()
    root.title("Image and Video Search")
    root.geometry("800x600")

    # Intuit QuickBooks logo placeholder
    try:
        logo = Image.open("Intuit_QuickBooks_logo.png")  # Replace with the actual logo file path
        max_width, max_height = 500, 500
        aspect_ratio = min(max_width / logo.width, max_height / logo.height)
        new_size = (int(logo.width * aspect_ratio), int(logo.height * aspect_ratio))
        logo = logo.resize(new_size)
        logo_img = ImageTk.PhotoImage(logo)
        logo_label = tk.Label(root, image=logo_img)
        logo_label.image = logo_img  # Keep a reference to avoid garbage collection
        logo_label.place(relx=0.5, rely=0.2, anchor="center")
    except FileNotFoundError:
        logo_label = tk.Label(root, text="QuickBooks Logo", font=("Helvetica", 20))
        logo_label.place(relx=0.5, rely=0.2, anchor="center")

    # Subtitles
    subtitle = tk.Label(root, text="Image Search", font=("Helvetica", 18))
    subtitle.place(relx=0.6, rely=0.3, anchor="center")

    # URL input
    url_label = tk.Label(root, text="Enter URL:")
    url_label.place(relx=0.5, rely=0.4, anchor="center")
    url_input = ttk.Entry(root, width=60)
    url_input.place(relx=0.5, rely=0.45, anchor="center")

    # Text input
    text_label = tk.Label(root, text="Enter text for similarity search:")
    text_label.place(relx=0.5, rely=0.5, anchor="center")
    text_input = tk.Text(root, height=3, width=60)
    text_input.place(relx=0.5, rely=0.55, anchor="center")

    # Search button
    search_button = ttk.Button(root, text="Search", command=search_action)
    search_button.place(relx=0.5, rely=0.65, anchor="center")

    # Results frame
    global results_frame
    results_frame = tk.Frame(root)
    results_frame.place(relx=0.5, rely=0.8, anchor="center")

    root.mainloop()

# Run the GUI
if __name__ == "__main__":
    create_gui()


Failed to fetch URL: 403 Client Error: Forbidden for url: https://www.yelp.com/biz/stones-landscaping-san-jose
